In [ ]:
from __future__ import print_function
import os

# Temporary workaround for Windows OpenMP duplicate runtime initialization in notebooks.
# Must be set before importing torch/torchvision/numpy-linked libraries.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Set up absolute path to the root of the project for importing custom modules.
sys.path.append(os.path.abspath("../.."))

import numpy as np
import torch as torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision.datasets import ImageFolder
import matplotlib as plt
import pickle

from utils.train_val_utils import train_and_evaluate
from utils.test_utils import evaluate_split
from models.ViT.ViT import VisionTransformerProcessor, ViT

torch.manual_seed(42)
use_cuda = True
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")
print(f"Using device: {device}")

CUDA available: True
Using device: cuda


In [ ]:
torch.manual_seed(42)
def flip_target(y): 
    return 1 - y

transforms_resize = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = ImageFolder(root="../../butterfly_anomaly_image_resized/final_dataset/train", target_transform=flip_target, transform=transforms_resize)
print("Classes:", train_dataset.classes)
print("Class->idx:", train_dataset.class_to_idx)
print("Total:", len(train_dataset))

val_dataset = ImageFolder(root="../../butterfly_anomaly_image_resized/final_dataset/val", target_transform=flip_target, transform=transforms_resize)
print("Classes:", val_dataset.classes)
print("Class->idx:", val_dataset.class_to_idx)
print("Total:", len(val_dataset))
test_dataset = ImageFolder(root="../../butterfly_anomaly_image_resized/final_dataset/test", target_transform=flip_target, transform=transforms_resize)
print("Classes:", test_dataset.classes)
print("Class->idx:", test_dataset.class_to_idx)
print("Total:", len(test_dataset))

Classes: ['0_non-hybrid', '1_hybrid']
Class->idx: {'0_non-hybrid': 0, '1_hybrid': 1}
Total: 3538
Classes: ['0_non-hybrid', '1_hybrid']
Class->idx: {'0_non-hybrid': 0, '1_hybrid': 1}
Total: 442
Classes: ['0_non-hybrid', '1_hybrid']
Class->idx: {'0_non-hybrid': 0, '1_hybrid': 1}
Total: 444


In [ ]:
torch.manual_seed(42)
#Add Augmented Images to Dataset
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

num_aug = 3538
# Rebuild datasets from the SAME split folders, only changing transforms
aug_train_dataset = ImageFolder(root="../../butterfly_anomaly_image_resized/final_dataset/train", transform=transform_train)
# aug_subset = Subset(aug_train_dataset, np.random.randint(0, len(aug_train_dataset), size=(num_aug)))
act_train_aug_dataset = ConcatDataset([train_dataset, aug_train_dataset]) #Combine both normal and aug  
print(len(train_dataset), len(aug_train_dataset), len(act_train_aug_dataset))

act_train_aug_dataset.class_to_idx = aug_train_dataset.class_to_idx
act_train_aug_dataset.classes = aug_train_dataset.classes

# Optional sanity check: class mapping must match across splits
assert aug_train_dataset.class_to_idx == val_dataset.class_to_idx == test_dataset.class_to_idx == act_train_aug_dataset.class_to_idx

3538 3538 7076


## Models


In [36]:
torch.manual_seed(42)
# Constant parameters
img_size = 224
num_classes = 2
batch_size = 16

# Parameters for model
patch_size = 8
num_patches = (img_size//patch_size)**2 
embed_dim = 64
num_heads = 2
num_layers = 2
mlp_dim = 128
class_weight = 10

# Hyperparameters
epochs = 100
lr = 5e-3
w_decay = 1e-5
dropout = 0.1


#w_decy 1e-5 better than 1e-4
vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_init = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_init.parameters(), lr = lr, weight_decay=w_decay)
print(model_init)

ViT(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (mlp_head): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [ ]:
## INITIAL MODEL 
# ---------------------------
print("="*20)
print("Initial Model Training")
print("="*20)
results_init = train_and_evaluate(model_init, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_init.pkl', device=device, num_workers=0, embeddings = vi_processor.process_images)

Initial Model Training
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.3245, Val Loss: 0.4822, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.3245, Val Loss: 0.4822, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK INITIAL MODEL 
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_init.pkl"
hist = model_init.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("F2 over time:", [i["metrics"]["val_f2_per_class_history"] for i in valid[-5:]])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

NameError: name 'model_init' is not defined

In [ ]:
# Parameters for model
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 2
num_layers = 2
mlp_dim = 128
class_weight = 15

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.1

#w_decy 1e-5 better than 1e-4
vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_1 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_1.parameters(), lr = lr, weight_decay=w_decay)
# print(model_1)
#Test Evaluate
print("="*20)
print("Model 1 Training")
print("Changes Made: learning rate 5e-3 -> 1e-4, class weight 10 -> 15" )
print("="*20)
results_init = train_and_evaluate(model_1, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_1.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 1 Training
Changes Made: learning rate 5e-3 -> 1e-4, class weight 10 -> 15
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.1683, Val Loss: 0.5916, Train F1-Macro: 0.7140, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.6450652 0.7829652], Val F1-Per-Class: [0.       0.908642], Train F2-Macro: 0.7140, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.5417918 0.8862652], Val F2-Per-Class: [0.        0.9598331], No improvement: 4/6
Epoch [5/100], Train Loss: 0.1683, Val Loss: 0.5916, Train F1-Macro: 0.7140, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.6450652 0.7829652], Val F1-Per-Class: [0.       0.908642] Train F2-Macro: 0.7140, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.5417918 0.8862652], Val F2-Per-Class: [0.        0.9598331]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 1
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_1.pkl"
hist = model_1.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("F2 over time:", [i["metrics"]["val_f2_per_class_history"] for i in valid[-5:]])
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
F2 over time: [[array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9598331], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9598331], dtype=float32), array([0.       , 0.9598331], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0

In [ ]:
# Parameters for model
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 2
num_layers = 2
mlp_dim = 128
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.1

#w_decy 1e-5 better than 1e-4
vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_2 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_2.parameters(), lr = lr, weight_decay=w_decay)

print("="*20)
print("Model 2 Training")
print("Changes Made: class weight 15 -> 5" )
print("="*20)
#Test Evaluate
results_init = train_and_evaluate(model_2, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_2.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 2 Training
Changes Made: class weight 15 -> 5
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.2967, Val Loss: 0.5051, Train F1-Macro: 0.6449, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.5322848 0.7575107], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.6509, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.416289  0.8854219], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.2967, Val Loss: 0.5051, Train F1-Macro: 0.6449, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.5322848 0.7575107], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.6509, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.416289  0.8854219], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 2
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_2.pkl"
hist = model_2.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("F2 over time:", [i["metrics"]["val_f2_per_class_history"] for i in valid[-5:]])
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
F2 over time: [[array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9598331], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9598331], dtype=float32), array([0.       , 0.9619395], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9598331], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32)], [array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0.       , 0.9619395], dtype=float32), array([0

In [ ]:
# Parameters for model
img_size = 224
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_classes = 2
num_heads = 2
num_layers = 2
mlp_dim = 128
batch_size = 16
class_weight = 5

# Hyperparameters
epochs = 200
lr = 1e-5
w_decay = 1e-5
dropout = 0.1

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_3 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_3.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 3 Training")
print("Changes Made: lr 1e-4 -> 1e-5" )
print("="*20)
results_init = train_and_evaluate(model_3, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_3.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 3 Training
Changes Made: lr 1e-4 -> 1e-5
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/200], Train Loss: 0.4448, Val Loss: 0.4695, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/200], Train Loss: 0.4448, Val Loss: 0.4695, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 3
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_3.pkl"
hist = model_3.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.4821113278042382 and 0.4596792885235378
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
# Parameters for model
img_size = 224
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_classes = 2
num_heads = 2
num_layers = 2
mlp_dim = 128
batch_size = 16
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-6
w_decay = 1e-5
dropout = 0.1

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_4 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_4.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 4 Training")
print("Changes Made: lr 1e-4 -> 1e-6" )
print("="*20)
results_init = train_and_evaluate(model_4, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_4.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 4 Training
Changes Made: lr 1e-4 -> 1e-6
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.4778, Val Loss: 0.4534, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.4778, Val Loss: 0.4534, Train F1-Macro: 0.3333, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.        0.6666667], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.4167, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.        0.8333333], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 4
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_4.pkl"
hist = model_4.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.5445265973125569 and 0.4830229048218046
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
# Parameters for model
img_size = 224
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_classes = 2
num_heads = 2
num_layers = 2
mlp_dim = 128
batch_size = 16
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.2

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_5 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_5.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 5 Training")
print("Changes Made: dropout 0.1 -> 0.2" )
print("="*20)
results_init = train_and_evaluate(model_5, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_5.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 5 Training
Changes Made: dropout 0.1 -> 0.2
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.3264, Val Loss: 0.4828, Train F1-Macro: 0.7914, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.7766459 0.8060726], Val F1-Per-Class: [0.       0.908642], Train F2-Macro: 0.7910, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.7428306 0.8392435], Val F2-Per-Class: [0.        0.9598331], No improvement: 4/6
Epoch [5/100], Train Loss: 0.3264, Val Loss: 0.4828, Train F1-Macro: 0.7914, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.7766459 0.8060726], Val F1-Per-Class: [0.       0.908642] Train F2-Macro: 0.7910, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.7428306 0.8392435], Val F2-Per-Class: [0.        0.9598331]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 5
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_5.pkl"
hist = model_5.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.44889816561078916 and 0.5017958616039583
Best val F2 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
# Parameters for model
img_size = 224
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_classes = 2
num_heads = 2
num_layers = 2
mlp_dim = 128
batch_size = 16
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.4

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_6 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_6.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 6 Training")
print("Changes Made: dropout 0.1 -> 0.4" )
print("="*20)
results_init = train_and_evaluate(model_6, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_6.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 6 Training
Changes Made: dropout 0.1 -> 0.4
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.3250, Val Loss: 0.5332, Train F1-Macro: 0.6822, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.6091743 0.7552287], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.6836, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.5166833  0.85059017], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.3250, Val Loss: 0.5332, Train F1-Macro: 0.6822, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.6091743 0.7552287], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.6836, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.5166833  0.85059017], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 6
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_6.pkl"
hist = model_6.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.48261236845101513 and 0.4803738814911672
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
# Constant Parameters
img_size = 224
num_classes = 2
batch_size = 16

# Parameters for model
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 4
num_layers = 4
mlp_dim = 128
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.4

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_7 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_7.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 7 Training")
print("="*20)
results_init = train_and_evaluate(model_7, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_7.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 7 Training
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.2882, Val Loss: 0.5704, Train F1-Macro: 0.7671, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.7173405 0.8168161], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.7650, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.616849  0.9132337], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.2882, Val Loss: 0.5704, Train F1-Macro: 0.7671, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.7173405 0.8168161], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.7650, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.616849  0.9132337], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 7
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_7.pkl"
hist = model_7.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.4699590882584957 and 0.5374237479908126
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
## MODEL 8
# -------------------------------------

# Parameters for model
patch_size = 8
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 4
num_layers = 2
mlp_dim = 128
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.4

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_8 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_8.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 8 Training")
print("="*20)
results_init = train_and_evaluate(model_8, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_8.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 8 Training
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.1873, Val Loss: 0.8039, Train F1-Macro: 0.5505, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.37818015 0.7228522 ], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.5712, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.2754223  0.86702937], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.1873, Val Loss: 0.8039, Train F1-Macro: 0.5505, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.37818015 0.7228522 ], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.5712, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.2754223  0.86702937], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 8
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_8.pkl"
hist = model_8.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.2696557569550876 and 0.5780284285013165
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
## MODEL 9
# -------------------------------------
# Parameters for model
patch_size = 4
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 4
num_layers = 4 
mlp_dim = 128
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.4

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_9 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_9.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 9 Training")
print("="*20)
results_init = train_and_evaluate(model_9, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_9.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 9 Training
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.1351, Val Loss: 0.5885, Train F1-Macro: 0.8117, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.7798571 0.8436065], Val F1-Per-Class: [0.       0.908642], Train F2-Macro: 0.8094, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.6948824 0.923867 ], Val F2-Per-Class: [0.        0.9598331], No improvement: 4/6
Epoch [5/100], Train Loss: 0.1351, Val Loss: 0.5885, Train F1-Macro: 0.8117, Val F1-Macro: 0.4543, Train F1-Per-Class: [0.7798571 0.8436065], Val F1-Per-Class: [0.       0.908642] Train F2-Macro: 0.8094, Val F2-Macro: 0.4799, Train F2-Per-Class: [0.6948824 0.923867 ], Val F2-Per-Class: [0.        0.9598331]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 9
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_9.pkl"
hist = model_9.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.26222205387415787 and 0.5271916861778924
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191


In [ ]:
## MODEL 10
# -------------------------------------
# Parameters for model
patch_size = 16
embed_dim = 64
num_patches = (img_size//patch_size)**2
num_heads = 4
num_layers = 4
mlp_dim = 128
class_weight = 5

# Hyperparameters
epochs = 100
lr = 1e-4
w_decay = 1e-5
dropout = 0.4

vi_processor = VisionTransformerProcessor(img_size = img_size, patch_size = patch_size, embed_dim = embed_dim, device = device)

model_10 = ViT(embed_dim = embed_dim, num_patches = num_patches, num_classes = num_classes, 
            num_heads = num_heads, num_layers = num_layers, mlp_dim = mlp_dim, dropout=dropout).to(device)
optimizer = optim.Adam(model_10.parameters(), lr = lr, weight_decay=w_decay)

#Test Evaluate
print("="*20)
print("Model 10 Training")
print("Changes Made: patch_size 8 -> 16" )
print("="*20)
results_init = train_and_evaluate(model_10, train_set=act_train_aug_dataset, val_set=val_dataset, 
optimizer=optimizer, num_epochs = epochs, batch_size = batch_size, class_weights_val = class_weight, 
    ckpt_file = '../../saved_models/vit_a/vit_a_10.pkl', device=device, embeddings = vi_processor.process_images, num_workers=0)

Model 10 Training
Changes Confirmed: class_weight -> 15, dropout -> 0.2, learning_rate -> 1e-4
Changes Made: patch_size 8 -> 16
  [Epoch 1] Improvement! New best score: 0.961940
Epoch [5/100], Train Loss: 0.1729, Val Loss: 0.7629, Train F1-Macro: 0.5725, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.41478163 0.73025703], Val F1-Per-Class: [0.        0.9099877], Train F2-Macro: 0.5891, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.30704954 0.87113935], Val F2-Per-Class: [0.        0.9619395], No improvement: 4/6
Epoch [5/100], Train Loss: 0.1729, Val Loss: 0.7629, Train F1-Macro: 0.5725, Val F1-Macro: 0.4550, Train F1-Per-Class: [0.41478163 0.73025703], Val F1-Per-Class: [0.        0.9099877] Train F2-Macro: 0.5891, Val F2-Macro: 0.4810, Train F2-Per-Class: [0.30704954 0.87113935], Val F2-Per-Class: [0.        0.9619395]

Early stopping at epoch 7. No improvement for 6 epochs.
Restored model weights from best epoch (score: 0.961940)


In [ ]:
## CHECK  MODEL 10
# -------------------------------------------
ckpt_file = "../../saved_models/vit_a/vit_a_10.pkl"
hist = model_10.load_checkpoint_history(ckpt_file)

valid = [
r for r in hist
if r.get("metrics", {}).get("val_f2_per_class_history")
and len(r["metrics"]["val_f2_per_class_history"]) > 0
]

best = max(
valid,
key=lambda r: float(r["metrics"]["val_f2_per_class_history"][-1][1])
)

print(len(valid))
print("Selected epoch:", best["epoch"])
print("Best Train & Val Loss: {} and {}".format(float(best["metrics"]["train_loss_history"][-1]), float(best["metrics"]["val_loss_history"][-1])))
print("Best val F2 (class 0):", float(best["metrics"]["val_f2_per_class_history"][-1][0]))
print("Best val F2 (class 1):", float(best["metrics"]["val_f2_per_class_history"][-1][1]))

7
Selected epoch: 1
Best Train & Val Loss: 0.26353196115243516 and 0.527021165538047
Best val F1 (class 0): 0.0
Best val F2 (class 1): 0.9619395136833191
